# Lecture 2a - Bayesian coding

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/stat_rethinking_2022

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 200)
    invisible(dev.off())
}

## Kalahari foragers example

Let's import the Nancy Howell's data from the Kalahari people and take a look:

In [ ]:
# Import data
xdata <- read.csv('../data/howell.csv')
# Display top 5 rows
head(xdata, 5)

In [ ]:
# Table of descriptive statistics
summary(xdata)

In [ ]:
# Remove kids from dataframe
kdata <- xdata[xdata$age > 17, ]
summary(kdata)

As a first example, let's take a look at the distribution of the data

In [ ]:
# Plot the distribution (kernel density) of adult heights
plot(density(kdata$height), main = '!Kung heights by Nancy Howell', xlab = 'Height (cm)')
# savefig('kungheight.jpg')

We can start our model with a basic (null) model, using a normal distribution to summarize the distribution of adult heights:

$$
\large{
h_i \sim Normal(\mu,\sigma)
}
$$


This describes the likelihood (or data distribution) part of the model, which being normal has two parameters that need priors:

$$
\large{
\begin{align*}
h_i &\sim Normal(\mu,\sigma) \\
\mu &\sim Normal(178, 20) \\
\sigma &\sim Uniform(0, 50)
\end{align*}}
$$

or again, more succinctly,

$$
\large{
\begin{align*}
h_i &\sim N(\mu,\sigma) \\
\mu &\sim N(178, 20) \\
\sigma &\sim U(0, 50)
\end{align*}}
$$

We can plot the distribution of these priors to see what they assume:

In [ ]:
# Setup multipanel figure
options(repr.plot.width = 10, repr.plot.height = 5)
par(mfrow = c(1, 2))

# Plot range of normal prior
x <- seq(100, 250, length.out = 100)
plot(x, dnorm(x, 178, 20), type = "l", xlab = 'μ', ylab = "Density", main = 'μ~N(178,20)')

# Plot range of sigma prior
x <- seq(-10, 60, length.out = 100)
plot(x, dunif(x, 0, 50), type = "l", xlab = 'σ', ylab = "Density", main = 'σ~U(0, 50)')

# savefig('kungpriors.jpg', width = 10)
par(mfrow = c(1, 1))
options(repr.plot.width = 7, repr.plot.height = 5)

So now we have a model - a likelihood and some priors - from which we can simulate, even though we have yet to see any data. To do this, we'll draw 1000 samples from our priors, then fire them into a normal and store the value at each iteration:


In [ ]:
# Number of samples
nsamp <- 1000
# Grab samples from
mu_ <- rnorm(nsamp, 178, 20)
sigma_ <- runif(nsamp, 0, 50)
h_ <- rnorm(nsamp, mu_, sigma_)
plot(density(h_), main = 'hi~N(μ,σ) Prior predictive distribution', xlab = 'Height (cm)', yaxt = "n", ylab = "")
# savefig('kungpriorsim.jpg')

What this gives us is some idea about how realistic our results are in terms of the **a priori** allowable height values for adults. Are these reasonable? Well we could add lines to indicate some known information - the tallest ever person ([Robert Wadlow](https://en.wikipedia.org/wiki/Robert_Wadlow)) who topped out at 272 cm:

In [ ]:
# Number of samples
nsamp <- 1000
# Grab samples from
mu_ <- rnorm(nsamp, 178, 20)
sigma_ <- runif(nsamp, 0, 50)
h_ <- rnorm(nsamp, mu_, sigma_)
plot(density(h_), main = 'hi~N( N(178, 20) , U(0, 50)) Prior predictive distribution',
     xlab = 'Height (cm)', yaxt = "n", ylab = "", xlim = c(min(h_), 330))
abline(v = 272)
text(275, 0.010, "Robert \n Wadlow \n (8'11'')", adj = 0)
text(275, 0.0011, "P(>Wadlow) \n =0.01", adj = 0)
# savefig('kungpriorsim2.jpg')

Which amounts to 1.3% percent of our a priori people being taller than the tallest ever person:

In [ ]:
mean(h_ > 272)

What about those wide 'uninformative' priors I've heard so much about? Well, this poses a problem for heights, which by definition can't go below zero. If we use an $N(178,1000)$ prior, this is what happens:

In [ ]:
# Number of samples
nsamp <- 1000
# Grab samples from
mu_ <- rnorm(nsamp, 178, 1000)
sigma_ <- runif(nsamp, 0, 50)
h_ <- rnorm(nsamp, mu_, sigma_)
plot(density(h_), main = 'hi~N( N(178, 1000) , U(0, 50)) Prior predictive distribution',
     xlab = 'heights', yaxt = "n", ylab = "")
abline(v = 272)
text(300, 0, "P(>Wadlow) \n =0.5", adj = c(0, 0))
# savefig('kungpriorsim3.jpg')

In [ ]:
mean(h_ > 272)

So both the mean **and the variance** of our priors matter. We should be skeptical of them, and test their implications before we hit the inference button. Given our original $N(178,20)$ prior, let's use the grid approximation for one last time to see what our likelihood surface looks like. To do this we can use the handy `expand.grid()` function, which builds every combination of the values we give it. To take a look at what it does we can use a `?`:

In [ ]:
?expand.grid

With this, we can then evaluate all the combinations of μ and σ across our entire grid:

In [ ]:
# Build grid from 100 to 259, against 4 to 9, in steps of 1 (μ in the first column, σ in the second)
pgrid <- as.matrix(expand.grid(sigma = 4:9, mu = 100:259)[, c("mu", "sigma")])
head(pgrid)

In [ ]:
# Look at grid
options(repr.plot.width = 20, repr.plot.height = 5)
plot(NA, xlim = c(100, 260), ylim = c(4, 9), xlab = "", ylab = "", xaxs = "i", yaxs = "i")
abline(v = 100:260, h = 4:9, col = "grey")
# savefig('gridx.jpg', width = 20)
options(repr.plot.width = 7, repr.plot.height = 5)

In [ ]:
# Build grid from 100 to 259, against 4 to 9, in steps of 1
pgrid <- as.matrix(expand.grid(sigma = 4:9, mu = 100:259)[, c("mu", "sigma")])
pgrid

Let's start with the priors and calcualte their likelihoods (confusing name alert), starting with the first two pairs of values on the grid

In [ ]:
# Calculate prior likelihood for first pair of values on the grid
c(dnorm(pgrid[1,1], 178, 20, log = TRUE), dunif(pgrid[1,2], 0, 50, log = TRUE))

In [ ]:
dnorm(pgrid[1,1], 178, 20, log = TRUE) + dunif(pgrid[1,2], 0, 50, log = TRUE)

In [ ]:
# Calculate prior likelihood for second pair of values on the grid
c(dnorm(pgrid[2,1], 178, 20, log = TRUE), dunif(pgrid[2,2], 0, 50, log = TRUE))

In [ ]:
# Calculate prior likelihood for last pair of values on the grid
n_grid <- nrow(pgrid)
c(dnorm(pgrid[n_grid,1], 178, 20, log = TRUE), dunif(pgrid[n_grid,2], 0, 50, log = TRUE))

In [ ]:
# Sum
dnorm(pgrid[n_grid,1], 178, 20, log = TRUE) + dunif(pgrid[n_grid,2], 0, 50, log = TRUE)

In [ ]:
# Calculate priors
prior_loglike <- dnorm(pgrid[,1], 178, 20, log = TRUE) + dunif(pgrid[,2], 0, 50, log = TRUE)
prior_loglike

We typically call these values the **prior probabilty** (likelihood here though, as we haven't standardized), so what do they look like?

In [ ]:
# Create a mu by sigma grid
X <- 100:259
Y <- 4:9
# Calculate prior likelihood on grid (a matrix with rows = μ, columns = σ)
Z <- outer(X, Y, function(m, s) dnorm(m, 178, 20, log = TRUE) + dunif(s, 0, 50, log = TRUE))

# Helper to colour the facets of a surface plot by height (blue = low, red = high)
facet_cols <- function(z, n = 100){
    zf <- (z[-1, -1] + z[-1, -ncol(z)] + z[-nrow(z), -1] + z[-nrow(z), -ncol(z)])/4
    hcl.colors(n, "Blue-Red")[cut(zf, n)]
}

# Plot the surface
persp(X, Y, Z, theta = -45, phi = 25, col = facet_cols(Z), border = NA, ticktype = "detailed",
      xlab = 'μ', ylab = 'σ', zlab = "", main = 'Prior likelihood')
# savefig('priors.jpg')

We get a flat top because in the σ dimension we have a uniform distribution; although odd, this is a 'heat' contour, so it's 'red hot' peak is at 178, regardless of the value of σ.

Ok, we have the prior likelihood, now we need the data likelihood - substituting in the grid pair values for the mean and standard deviation for each recorded height. Let's start with the first pair on the grid and the first height value:

In [ ]:
xdata$height[1]

In [ ]:
dnorm(kdata$height[1], mean = pgrid[1,1], sd = pgrid[1,2], log = TRUE)

In [ ]:
# Calculate for first pair of values on the grid
sum(dnorm(kdata$height, mean = pgrid[1,1], sd = pgrid[1,2], log = TRUE))

In [ ]:
# Calculate for last pair of values on the grid
sum(dnorm(kdata$height, mean = pgrid[n_grid,1], sd = pgrid[n_grid,2], log = TRUE))

In [ ]:
# Calculate likelihood (data distribution) for a normal (on the log scale) for each point in the grid,
# using sapply to loop over the rows of the grid
log_likelihood <- sapply(1:n_grid, function(i) sum(dnorm(kdata$height, mean = pgrid[i,1], sd = pgrid[i,2], log = TRUE)))
log_likelihood

And we can take a look at the **likelihood surface**, which is where frequentist analysis stops 

In [ ]:
# Calculate data likelihood on grid
Zll <- outer(X, Y, Vectorize(function(m, s) sum(dnorm(kdata$height, m, s, log = TRUE))))

# Plot the surface
persp(X, Y, Zll, theta = -45, phi = 25, col = facet_cols(Zll), border = NA, ticktype = "detailed",
      xlab = 'μ', ylab = 'σ', zlab = "", main = 'Data likelihood')
# savefig('likelihood_surface.jpg')

In [ ]:
# Maximum likelihood estimate
list(pgrid[log_likelihood == max(log_likelihood), ], max(log_likelihood))

Now that we have log-scale data likelhood and prior probability values, we can **add** them together to calculate the numerator of Bayes theorem:

$$
\text{Posterior} = \frac{\text{likelihood x prior}}{\text{normalizing constant}}
$$

In [ ]:
# Un-normalized posterior
post_num <- log_likelihood + prior_loglike

In [ ]:
# Normalized posterior - remember we're doing log-probability calculations here, so don't foget to exponentiate!
# Also in log-land, the division becomes subtraction, and the normalization is relative to the largst value (for plotting)
posterior <- exp(post_num - max(post_num))

With this, we can take a look at what the posterior surface looks like:

In [ ]:
# Put posterior values back into a μ (rows) by σ (columns) matrix
zi <- matrix(posterior, nrow = length(X), ncol = length(Y), byrow = TRUE)
# Contour plot of results
contour(X, Y, zi, xlab = 'μ', ylab = 'σ', main = 'Posterior')
# savefig('posterior.jpg')

Let's zoom in

In [ ]:
# Contour plot of results
contour(X, Y, zi, xlab = 'μ', ylab = 'σ', main = 'Posterior', xlim = c(150, 160), ylim = c(7, 9))
# savefig('posterior_zoom.jpg')

So this is what's known as the **joint posterior** density for μ and σ, evaluated using grid approximation. 

So congradulations - we've calcualted the mean and standard deviation of some data, but in a really important way (*i.e.* using Bayes theorem). It's important to keep in mind that while means and variacnes are things we can easily calculate using math (hence their widespread use), they still constitute a fully Bayesian model for the height data. **Plus** we get uncertainty estimates for μ and σ, which is important. 

Next we'll add some complexity to our model - a covariate of weight - to do what a more typical kind of linear regression. First, let's see what sort of relationship we have between these variables:

In [ ]:
plot(kdata$weight, kdata$height, xlab = 'Weight (kg)', ylab = 'Height (cm)')
# savefig('scatter.jpg')

Ok, looks linear enough. Now let's write out a linear model for this:

$$
\large{
\begin{align*}
h_i &\sim N(\mu_i,\sigma) \\
\mu_i &= \beta_0 + \beta_1(x_i - \bar{x}) \\
\beta_0 &\sim N(178,20) \\
\beta_1 &\sim N(0,10) \\
\sigma &\sim U(0, 50)
\end{align*}}
$$

So, as before, let's simulate and see if we have reasonable priors. This time we are simulating possible lines to describe the relationship between weight and height.

In [ ]:
# Number of samples
nsamp <- 100
# Intercept - estimated height at popultion average weight
b0_ <- rnorm(nsamp, 178, 20)
# Slope - relationship between weight and height
b1_ <- rnorm(nsamp, 0, 10)

In [ ]:
# Grab range of weights to plot over
meanweight <- mean(kdata$weight)
weights_ <- seq(min(kdata$weight), max(kdata$weight), length.out = 50) - meanweight

# Empty plot, then add lines given sample values for β0 and β1
plot(NA, xlim = range(weights_ + meanweight), ylim = c(-100, 400),
     xlab = 'weight (kg)', ylab = 'height (cm)', main = 'β1~N(0,10) Prior predictive simulation')
for (i in 1:nsamp) lines(weights_ + meanweight, b0_[i] + b1_[i]*weights_, col = adjustcolor("black", 0.1))

# Add min and max human heights
abline(h = c(0, 272), col = "dodgerblue")
# savefig('priorPlinear.jpg')

So we know that people can't be less than zero or more than 272 cm tall, so our model is a priori quite wrong, and we have the opportunity to do a bit better, using our 'domain knowledge' (external information, expertise, common sense etc.) to do a bit better. Given that we know that the relationship between weight and height in people is positive, we can do a bit better. McElreath suggests a log-normal prior because it is by definition constrained to be above zero. Let's try a $log-Normal(0, 1)$ prior and see what happens:

In [ ]:
# New slope prior for the relationship between weight and height
b1_ <- rlnorm(nsamp, 0, 1)

In [ ]:
# Plot new lines given sample values for β0 and β1
plot(NA, xlim = range(weights_ + meanweight), ylim = c(-100, 400),
     xlab = 'weight (kg)', ylab = 'height (cm)', main = 'β1~logN(0,1) Prior predictive simulation')
for (i in 1:nsamp) lines(weights_ + meanweight, b0_[i] + b1_[i]*weights_, col = adjustcolor("black", 0.1))

# Add min and max human heights
abline(h = c(0, 272), col = "dodgerblue")
# savefig('priorPlinear2.jpg')

So while there are some extreme values still possible for max heights, the lower values are constrained to be positive, which is much better. Incidentally plotting a hundred or more lines from some distribtuion is a very Bayesian way to get a look at uncertainty - they tend to illustrate odd cases such as above and are **way** easier to plot than formal uncertainty intervals (credible intervals) around the mean trendline.

With reasonable priors in hand we can now use MCMC in Stan to calculate our posteriors:

In [ ]:
plot(kdata$weight - meanweight, kdata$height)

A Stan program is written in blocks: `data` declares what we observe, `parameters` declares what we want to learn about (with any constraints, like $\sigma$ being between 0 and 50), and `model` holds the priors and likelihood. Stan is a compiled language, so the first time we build a model it takes a little while to compile to C++.

In [ ]:
# Linear regression model in Stan
kalahari2_code <- "
data {
  int<lower=0> N;
  vector[N] weight;    // centred weights
  vector[N] height;
}
parameters {
  real Average_height;
  real<lower=0> Weight_slope;
  real<lower=0, upper=50> Obs_sd;
}
model {
  // Priors
  Average_height ~ normal(178, 20);
  Weight_slope ~ lognormal(0, 1);
  Obs_sd ~ uniform(0, 50);

  // Linear model
  vector[N] mu = Average_height + Weight_slope*weight;

  // Likelihood
  height ~ normal(mu, Obs_sd);
}
"
kalahari2 <- cmdstan_model(write_stan_file(kalahari2_code))

# Data as a named list
kdat <- list(N = nrow(kdata), weight = kdata$weight - meanweight, height = kdata$height)

# Sampling using MCMC (NUTS) in Stan
trace_k <- kalahari2$sample(data = kdat, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)

In [ ]:
hist(trace_k$draws("Average_height", format = "matrix"), main = "", xlab = "Average_height")

In [ ]:
mcmc_combo(trace_k$draws(c("Average_height", "Weight_slope", "Obs_sd")), combo = c("dens_overlay", "trace"))

In [ ]:
# Grab median value from the posterior of average height
b0 <- median(trace_k$draws("Average_height"))
# Grab median value from the posterior of the effect of weight on height
b1 <- median(trace_k$draws("Weight_slope"))
# Grab median value from the posterior of the variability around the line
sig <- median(trace_k$draws("Obs_sd"))

c(b0, b1, sig)

So we have some parameter estimates, lets see how this fits to our data:

In [ ]:
plot(kdata$weight, kdata$height, xlab = 'Weight (kg)', ylab = 'Height (cm)')
lines(weights_ + meanweight, b0 + b1*weights_, lwd = 2)
# savefig('fitline.jpg')

Looks not too bad, how about those uncertainty bounds?

In [ ]:
plot(kdata$weight, kdata$height, xlab = 'Weight (kg)', ylab = 'Height (cm)')
y_ <- b0 + b1*weights_
y_uu <- y_ + sig*2
y_ul <- y_ - sig*2
lines(weights_ + meanweight, y_, lwd = 2)
lines(weights_ + meanweight, y_uu, lty = 3, lwd = 2)
lines(weights_ + meanweight, y_ul, lty = 3, lwd = 2)
# savefig('fitlines.jpg')

And what do those uncertainty bounds represent?

In [ ]:
# Pick every 5th weight along the line at which to draw the normal likelihood
w_at <- weights_[seq(1, length(weights_), by = 5)] + meanweight
h_seq <- seq(-3*sig, 3*sig, length.out = 50)

# Figure: empty 3D box, then add the data, the line, and a normal curve at each weight
options(repr.plot.width = 10, repr.plot.height = 10)
pmat <- persp(range(kdata$weight), range(c(kdata$height, y_ + 3*sig, y_ - 3*sig)), matrix(0, 2, 2),
              zlim = c(0, dnorm(0, 0, sig)), theta = -30, phi = 20, border = NA, box = TRUE,
              xlab = 'Weight (kg)', ylab = 'Height (cm)', zlab = "", ticktype = "detailed")
points(trans3d(kdata$weight, kdata$height, 0, pmat), col = "dodgerblue", pch = 16, cex = 0.6)
lines(trans3d(weights_ + meanweight, y_, 0, pmat), lwd = 2)
for (w in w_at){
    mu_w <- b0 + b1*(w - meanweight)
    lines(trans3d(rep(w, 50), mu_w + h_seq, dnorm(h_seq, 0, sig), pmat), col = "steelblue4")
}
# savefig('wireframe.jpg', width = 10, height = 10)
options(repr.plot.width = 7, repr.plot.height = 5)

Et voilà! We have fit our second Bayesian linear regression model (the first that looks like a line). You should feel proud, this is a big foundation on which to build your skilz.